In [14]:
import sys
import pickle
import numpy as np
import pandas as pd
from itertools import combinations
from math import factorial
from collections import defaultdict
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tsa.stattools import adfuller
from statsmodels.regression.linear_model import RegressionResultsWrapper 

sys.path.append('..')
from modules.source import Sources, parquet_monthly, PROCESSED_DATA_PATH

CHANNELS: dict[str, tuple[str]] = {
  "Domestic Baseline": (Sources.cpi_inflation, Sources.IPI),
  "Rate Differential": (Sources.rate_differential,),
  "Global Risk Appetite": (Sources.VIX,),
  "Global Credit Condition": (Sources.UST_10Y, Sources.HY_OAS),
  "FX Pass-Through": (Sources.USDMYR,),
  "Commodity": (Sources.crude_palm_oil, Sources.brent_oil),
}
ALL_PREDICTORS = [v for vlist in CHANNELS.values() for v in vlist]
DEPENDENT_VARS = [Sources.KLCI, Sources.financials, Sources.energy, Sources.industrial_products, Sources.plantation, Sources.REITs, Sources.technology]

df_monthly = pd.read_parquet(parquet_monthly).xs("change", axis=1, level=1)

def fit_hac_model(target, rhs_vars, data) -> RegressionResultsWrapper:
  formula = f"{target} ~ {' + '.join(rhs_vars)}"
  L = int(np.floor(4 * (len(data) / 100) ** (2 / 9)))
  return smf.ols(formula, data=data).fit(cov_type="HAC", cov_kwds={"maxlags": L})

def shapley_r2(y: pd.Series, X: pd.DataFrame) -> dict[str, float]:
  predictors = list(X.columns)
  p = len(predictors)
  Xc = sm.add_constant(X)
  r2_cache: dict[frozenset, float] = {frozenset(): 0.0}
  for k in range(1, p + 1):
    for combo in combinations(predictors, k):
      r2_cache[frozenset(combo)] = sm.OLS(y, Xc[["const"] + list(combo)]).fit().rsquared
  shapley = {}
  for j in predictors:
    rest = [v for v in predictors if v != j]
    total = 0.0
    for k in range(len(rest) + 1):
      weight = factorial(k) * factorial(p - k - 1) / factorial(p)
      for combo in combinations(rest, k):
        s = frozenset(combo)
        total += weight * (r2_cache[s | {j}] - r2_cache[s])
    shapley[j] = total
  return shapley

full_models, fit_dict, partialr2_dict, beta_dict = {}, {}, {}, {}
wald_dict = defaultdict(dict)

for d_var in DEPENDENT_VARS:
  data = df_monthly[[d_var] + ALL_PREDICTORS].dropna()
  m_full = fit_hac_model(d_var, ALL_PREDICTORS, data)
  full_models[d_var] = m_full

  fit_dict[d_var] = {
    ("Full R2"): round(m_full.rsquared, 4),
    ("Adj. R2"): round(m_full.rsquared_adj, 4),
  }

  shap = shapley_r2(data[d_var], data[ALL_PREDICTORS])
  partialr2_dict[d_var] = {}
  beta_dict[d_var] = {}
  for ch_name, ch_vars in CHANNELS.items():
    for var in ch_vars:
      partialr2_dict[d_var][(ch_name, var)] = round(shap[var], 4)
      beta_dict[d_var][(ch_name, var)] = m_full.params[var]

  for ch_name, ch_vars in CHANNELS.items():
    hypothesis = ", ".join(f"{v} = 0" for v in ch_vars)
    w_res = m_full.wald_test(hypothesis, use_f=True, scalar=True)
    wald_dict[ch_name][(d_var, "F-Stat")] = round(float(w_res.statistic), 4)
    wald_dict[ch_name][(d_var, "p-value")] = round(float(w_res.pvalue), 4)

df_beta = pd.DataFrame(beta_dict)
df_fit = pd.DataFrame(fit_dict)
df_wald = pd.DataFrame(wald_dict).T
df_partialr2 = pd.DataFrame(partialr2_dict).div(df_fit.loc["Full R2"], axis=1) * 100

### 1. ADF (Stationary), VIF (Multicollinearity)

In [15]:
print("Diagnostic Assumptions (ADF, VIF):")

diagnostic_dict = {}
for col in ALL_PREDICTORS + DEPENDENT_VARS:
  stat, p = adfuller(df_monthly[col].dropna(), autolag="AIC", result_object=True)[:2]
  diagnostic_dict[col] = {
    ("ADF", "score"): round(float(stat), 4),
    ("ADF", "p-val"): round(float(p), 4)
  }

exog_matrix = full_models[DEPENDENT_VARS[0]].model.exog
exog_names = full_models[DEPENDENT_VARS[0]].model.exog_names
for i, var_name in enumerate(exog_names):
  if var_name != "Intercept":
    vif_score = variance_inflation_factor(exog_matrix, i)
    diagnostic_dict[var_name][("VIF", "score")] = round(float(vif_score), 4)

df_adf_vif = pd.DataFrame(diagnostic_dict).T
df_adf_vif.columns.names = ["Test", ""]
print(df_adf_vif)

Diagnostic Assumptions (ADF, VIF):
Test                     ADF             VIF
                       score   p-val   score
cpi_inflation        -9.3565  0.0000  1.2350
IPI                  -4.5511  0.0002  1.2611
FFR_OPR_diff         -4.7400  0.0001  1.1928
VIX                  -5.7063  0.0000  1.6315
UST_10Y             -11.1366  0.0000  1.3136
HY_OAS               -9.3741  0.0000  2.5676
USDMYR              -11.0840  0.0000  1.2665
CPO                  -9.1417  0.0000  1.0883
Oil                  -8.5630  0.0000  1.9657
KLCI                -13.1469  0.0000     NaN
financials          -10.4197  0.0000     NaN
energy              -11.0151  0.0000     NaN
industrial_products -10.0120  0.0000     NaN
plantation          -13.8698  0.0000     NaN
REITs               -14.8697  0.0000     NaN
technology           -4.6504  0.0001     NaN


### 4. MODEL FIT & VARIANCE DECOMPOSITION

In [16]:
print("Model Fit:")
print(df_fit.to_string())
print("\nChannel Variance Decomposition (Contribution to R²):")
print(df_partialr2.groupby(level=0, sort=False).sum().to_string())

Model Fit:
           KLCI  financials  energy  industrial_products  plantation   REITs  technology
Full R2  0.3190      0.2651  0.2368               0.1496      0.1201  0.1922      0.2338
Adj. R2  0.2662      0.2081  0.1775               0.0837      0.0367  0.1296      0.1744

Channel Variance Decomposition (Contribution to R²):
                              KLCI  financials     energy  industrial_products  plantation      REITs  technology
Domestic Baseline         3.573668    0.829876  21.030405             5.614973    8.742714   5.202914    1.668092
Rate Differential         1.567398    1.697473   1.351351             6.417112   20.233139  17.585848    2.480753
Global Risk Appetite     17.115987   18.068653   3.758446             6.951872    2.414654  10.926119   32.420873
Global Credit Condition  39.686520   34.138061  12.668919            56.149733   15.487094  46.722164   52.865697
FX Pass-Through          23.040752   31.610713   0.548986             6.016043    1.748543   3.225

### 5. INFERENCE & FEATURE IMPORTANCE

In [17]:
print("Channel Wald Joint Significance Tests:")
print(df_wald.to_string())

Channel Wald Joint Significance Tests:
                            KLCI         financials          energy         industrial_products         plantation           REITs         technology        
                          F-Stat p-value     F-Stat p-value  F-Stat p-value              F-Stat p-value     F-Stat p-value  F-Stat p-value     F-Stat p-value
Domestic Baseline         0.8151  0.4451     0.5405  0.5839  3.8223  0.0247              0.1086  0.8972     1.1251  0.3289  1.0046  0.3694     0.6313  0.5337
Rate Differential         0.7586  0.3856     0.0541  0.8164  0.3935  0.5317              1.5535  0.2151     2.8585  0.0942  2.3233  0.1302     0.0052  0.9428
Global Risk Appetite      1.0185  0.3150     0.9171  0.3402  0.9913  0.3215              0.1941  0.6603     0.0016  0.9682  0.1501  0.6992     1.9110  0.1695
Global Credit Condition   4.9985  0.0083     4.0893  0.0192  0.8500  0.4301              3.0552  0.0509     0.0145  0.9856  3.1761  0.0454     6.4062  0.0023
FX Pass-Throu

In [18]:
analysis_df: dict[str, pd.DataFrame] = {
   'df_beta': df_beta,
   'df_fit': df_fit,
   'df_wald': df_wald,
   'df_partialr2': df_partialr2,
   'df_adf_vif': df_adf_vif,
}

with open(f"{PROCESSED_DATA_PATH}/analysis_df.pkl", "wb") as f:
   pickle.dump({"analysis_df": analysis_df}, f)